# 03 · Måling

## Opsætning

In [ ]:
import sys
from pathlib import Path

# VS Code runs a notebook in its own folder: walk up to training/, the folder with pyproject.toml.
TRAINING = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "pyproject.toml").is_file() and (p / "wt_train").is_dir())
sys.path.insert(0, str(TRAINING))

import importlib
import json
import re
import subprocess
import tempfile
import textwrap

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display

from wt_train import harness, paths

## 20 klip fra den seneste kørsel

In [ ]:
# Eval runs are named by date and time, so the last name is the latest. Read only, never rerun here.
RUN = sorted(p.parent for p in paths.eval_runs_dir().glob("*/fleurs-da-dk-test.jsonl"))[-1]
rows = [json.loads(line) for line in (RUN / "fleurs-da-dk-test.jsonl").read_text(encoding="utf-8").splitlines()]
run_summary = json.loads((RUN / "summary.json").read_text(encoding="utf-8"))
summary = run_summary["results"]["fleurs-da_dk-test"]
clips = rows[:20]

print("Kørsel:", RUN.name)
print("Model: ", run_summary["model"])
print("Klip:  ", len(rows), "i kørslen,", len(clips), "her")

## Normalisering

In [ ]:
pairs = [harness.normalize_pair(r["reference"], r["hypothesis"]) for r in clips]
print("Som i kørslen:", pairs == [(r["ref_norm"], r["hyp_norm"]) for r in clips])

# The example: the clip with the most kinds of error, the shortest first.
ex = min(range(len(clips)), key=lambda i: (-sum(clips[i][k] > 0 for k in ("sub", "del", "ins")),
                                           clips[i]["ref_words"]))
print("\nFacit:     ", clips[ex]["reference"])
print("Edda:      ", clips[ex]["hypothesis"])
print("\nFacit, normaliseret:", pairs[ex][0])
print("Edda, normaliseret: ", pairs[ex][1])
Audio(filename=clips[ex]["wav"])

## Levenshtein-tabellen

In [ ]:
def align(ref, hyp):
    """D[i, j] = fewest edits that turn ref[:i] into hyp[:j]; then walk back from the corner."""
    D = np.zeros((len(ref) + 1, len(hyp) + 1), dtype=int)
    D[:, 0] = np.arange(len(ref) + 1)  # delete every reference word
    D[0, :] = np.arange(len(hyp) + 1)  # insert every hypothesis word
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            D[i, j] = min(D[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]), D[i - 1, j] + 1, D[i, j - 1] + 1)
    # On a tie: substitution, then deletion, then insertion (as wt_eval).
    ops, i, j = [], len(ref), len(hyp)
    while i or j:
        if i and j and ref[i - 1] == hyp[j - 1]:
            ops.append(("=", ref[i - 1], hyp[j - 1])); i, j = i - 1, j - 1
        elif i and j and D[i, j] == D[i - 1, j - 1] + 1:
            ops.append(("S", ref[i - 1], hyp[j - 1])); i, j = i - 1, j - 1
        elif i and D[i, j] == D[i - 1, j] + 1:
            ops.append(("D", ref[i - 1], "")); i -= 1
        else:
            ops.append(("I", "", hyp[j - 1])); j -= 1
    return D, ops[::-1]


ref, hyp = pairs[ex][0].split(), pairs[ex][1].split()
D, ops = align(ref, hyp)
print("Rækker: facit, kolonner: Edda. Nederste højre hjørne =", D[-1, -1], "fejl")
display(pd.DataFrame(D, index=["", *ref], columns=["", *hyp]))

## Ordene side om side

In [ ]:
for k in range(0, len(ops), 9):
    chunk = ops[k:k + 9]
    widths = [max(len(r), len(h), 1) for _, r, h in chunk]
    print("facit ", "  ".join((r or "·").ljust(w) for (_, r, _), w in zip(chunk, widths)))
    print("edda  ", "  ".join((h or "·").ljust(w) for (_, _, h), w in zip(chunk, widths)))
    print("      ", "  ".join(("" if op == "=" else op).ljust(w) for (op, _, _), w in zip(chunk, widths)), "\n")

print("S (erstattet):", sum(op == "S" for op, _, _ in ops), " D (slettet):", sum(op == "D" for op, _, _ in ops),
      " I (indsat):", sum(op == "I" for op, _, _ in ops), " ord i facit:", len(ref))
print("I kørslens jsonl:", {k: clips[ex][k] for k in ("sub", "del", "ins", "ref_words")})

## WER regnet tre gange

In [ ]:
refs, hyps = [r for r, _ in pairs], [h for _, h in pairs]
by_hand = [align(r.split(), h.split())[1] for r, h in pairs]
subs, dels, ins = (sum(op == kind for ops_ in by_hand for op, _, _ in ops_) for kind in "SDI")
N = sum(len(r.split()) for r in refs)
J = jiwer.process_words(refs, hyps)
W = [harness.word_errors(r.split(), h.split()) for r, h in pairs]
jsonl = [sum(r[k] for r in clips) for k in ("sub", "del", "ins", "ref_words")]

table = pd.DataFrame(
    [[subs, dels, ins, N, (subs + dels + ins) / N],
     [J.substitutions, J.deletions, J.insertions, J.hits + J.substitutions + J.deletions, J.wer],
     [sum(c.sub for c in W), sum(c.dele for c in W), sum(c.ins for c in W), sum(c.ref for c in W),
      harness.corpus_wer(W)],
     [*jsonl, sum(jsonl[:3]) / jsonl[3]]],
    index=["i hånden", "jiwer", "værkstedet", "kørslens jsonl"], columns=["S", "D", "I", "N", "WER"])
display(table)
print("Alle fire ens:", len(table.drop_duplicates()) == 1)

# All clips: jiwer breaks ties its own way, so S, D and I can differ while S + D + I and the WER agree.
J_all = jiwer.process_words([r["ref_norm"] for r in rows], [r["hyp_norm"] for r in rows])
print(f"\nAlle {len(rows)} klip, jiwer S/D/I:", J_all.substitutions, J_all.deletions, J_all.insertions,
      " jsonl S/D/I:", *(sum(r[k] for r in rows) for k in ("sub", "del", "ins")))
print("WER, jiwer:", J_all.wer, " kørslens summary.json:", summary["wer"])

## 95 %-interval med bootstrap

In [ ]:
errors = np.array([r["sub"] + r["del"] + r["ins"] for r in rows])
words = np.array([r["ref_words"] for r in rows])
rng = np.random.default_rng(0)


def bootstrap(errors, words, resamples=1000):
    """Draw the clips again with replacement; the middle 95 % of those WERs is the interval."""
    picks = rng.integers(0, len(errors), size=(resamples, len(errors)))
    wers = errors[picks].sum(axis=1) / words[picks].sum(axis=1)
    return wers, np.percentile(wers, [2.5, 97.5])


wers, ci = bootstrap(errors, words)
_, ci20 = bootstrap(errors[:20], words[:20])
counts = [harness.Counts(sub=r["sub"], dele=r["del"], ins=r["ins"], ref=r["ref_words"]) for r in rows]
print(f"WER, alle {len(rows)} klip:   {errors.sum() / words.sum():.4f}")
for label, (lo, hi) in [("i hånden", ci), ("værkstedet", harness.bootstrap_ci(counts)),
                        ("kørslens summary", summary["wer_ci95"]), ("kun de 20 klip", ci20)]:
    print(f"{label + ':':<22}[{lo:.4f}, {hi:.4f}]")

fig, ax = plt.subplots(figsize=(10, 3.5), layout="constrained")
ax.hist(wers, bins=40, color="#4a7bb7")
for x in ci:
    ax.axvline(x, color="black", linestyle="--")
ax.set(title=f"{len(wers)} genudtrækninger af de {len(rows)} klip, med 95 %-intervallet", xlabel="WER",
       ylabel="Antal")
plt.show()

## Brøken 29¾ før og nu

In [ ]:
OLD = "a6199e0c"  # run 7, the last normaliser before run 8
with tempfile.TemporaryDirectory() as tmp:
    package = Path(tmp) / "wt_eval_run7"
    package.mkdir()
    (package / "__init__.py").write_text("")
    wanted = ["normalize"]
    while wanted:  # normalize.py and every module it imports from its package
        name = wanted.pop()
        source = subprocess.run(["git", "-C", str(paths.REPO), "show", f"{OLD}:models/eval/wt_eval/{name}.py"],
                                capture_output=True, text=True, check=True).stdout
        (package / f"{name}.py").write_text(source, encoding="utf-8")
        wanted += [m for m in re.findall(r"^from \.(\w+) import", source, re.M) if not (package / f"{m}.py").exists()]
    sys.path.insert(0, tmp)
    old = importlib.import_module("wt_eval_run7.normalize")
    sys.path.remove(tmp)

fraction = next(r for r in rows if "29¾" in r["reference"])
texts = ["29¾", "29 trefjerdedele"]
display(pd.DataFrame({"før (kørsel 7)": [old.normalize(t) for t in texts],
                      "nu": [harness.normalize(t) for t in texts]}, index=texts))

old_counts = [harness.word_errors(old.normalize(r["reference"]).split(), old.normalize(r["hypothesis"]).split())
              for r in rows]
print("Klippet med 29¾:", fraction["id"])
print(textwrap.fill(fraction["reference"], 110, initial_indent="  facit: ", subsequent_indent=" " * 9))
print(textwrap.fill(fraction["hypothesis"], 110, initial_indent="  Edda:  ", subsequent_indent=" " * 9))
print("  fejl før:", old_counts[rows.index(fraction)].errors, " nu:", fraction["sub"] + fraction["del"] + fraction["ins"])
print(f"Hele kørslen, WER før: {harness.corpus_wer(old_counts):.4f}  nu: {summary['wer']:.4f}")